# SSD（PASCAL VOC 2007を用いた物体検出）

---
## 目的
物体検出の代表的な手法であるSSD（Single Shot MultiBox Detector）を，PASCAL VOC 2007データセットを用いて実装する．複数スケールの特徴マップとデフォルトボックス（Prior Box）を用いることで，1回のネットワーク推論で様々な大きさの物体を検出する仕組みについて理解する．

## モジュールのインポート
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q torchmetrics

import os
import zipfile
from time import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import VOCDetection
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.ops import box_iou, box_convert, nms
from torchvision.utils import draw_bounding_boxes
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込みます．VOCデータセットは，人物・乗り物・動物・室内物品など20クラスの物体に対して矩形領域（Bounding Box）が付与されたデータセットで，学習用（trainval）5011枚，評価用（test）4952枚の画像から構成されます．データセットの読み込みやDataLoaderの基本的な使い方については`02_dnn_simple_pytorch/existing_dataset.ipynb`を参照してください．

ここでは，Googleドライブに配置したVOCdevkit（2007）のzipファイルを`gdown`でダウンロードし，`torchvision.datasets.VOCDetection`で読み込みます．

In [ ]:
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
# 背景（物体なし）を0番として，クラスラベルは1〜20を割り当てる
CLASS_TO_IDX = {name: i + 1 for i, name in enumerate(VOC_CLASSES)}
n_class = len(VOC_CLASSES) + 1  # 背景を含めたクラス数
IMG_SIZE = 300

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)


def parse_voc_target(target):
    """VOCDetectionが返すXML由来のdictを，(boxes, labels)のTensorに変換する"""
    objects = target['annotation']['object']
    if isinstance(objects, dict):  # 物体が1つだけの場合はdictのまま返るため，リストに揃える
        objects = [objects]

    boxes, labels = [], []
    for obj in objects:
        bbox = obj['bndbox']
        boxes.append([float(bbox['xmin']), float(bbox['ymin']), float(bbox['xmax']), float(bbox['ymax'])])
        labels.append(CLASS_TO_IDX[obj['name']])

    return {'boxes': torch.tensor(boxes, dtype=torch.float32), 'labels': torch.tensor(labels, dtype=torch.long)}


class VOCDetectionDataset(torch.utils.data.Dataset):
    """VOCDetectionをラップし，画像のリサイズに合わせてボックス座標をスケールしたTensorを返すDataset"""
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCDetection(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.to_tensor = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, target = self.voc[idx]
        w, h = image.size
        target = parse_voc_target(target)

        # 元画像サイズ -> (img_size, img_size) へのリサイズに合わせてボックス座標をスケール
        scale = torch.tensor([self.img_size / w, self.img_size / h, self.img_size / w, self.img_size / h])
        target['boxes'] = target['boxes'] * scale

        image = self.to_tensor(image)
        return image, target


def collate_fn(batch):
    # 画像1枚あたりの物体数が異なるため，ボックス・ラベルはリストのまま扱う
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets


train_data = VOCDetectionDataset('trainval')
test_data = VOCDetectionDataset('test')

print(len(train_data), len(test_data))
print('クラス数（背景含む）:', n_class)

## SSD（Single Shot MultiBox Detector）
SSDは，1回のネットワーク推論だけで物体の位置とクラスを同時に予測する「1段階（single-shot）」の物体検出手法です．ネットワーク内の異なる深さの複数の特徴マップ（浅い層ほど解像度が高く小さい物体の検出に適し，深い層ほど解像度が低く大きい物体の検出に適する）それぞれに対して，「デフォルトボックス（Prior Box）」と呼ばれる矩形を格子状に複数配置し，各デフォルトボックスに対して「クラス（背景か物体か）」と「デフォルトボックスからのずれ（オフセット）」を同時に回帰します．

### バックボーンと特徴マップ
原論文のSSDはVGG16をバックボーンとしていますが，本ノートブックでは`torchvision_models.ipynb`で扱った事前学習済みResNet50をバックボーンとして使用します．ResNet50の`layer2`・`layer3`・`layer4`の出力に加えて，さらにダウンサンプリングする畳み込み層を3つ追加することで，300×300の入力画像に対して38×38・19×19・10×10・5×5・3×3・1×1という6段階の解像度の特徴マップを得ます．

In [ ]:
class SSDBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool, resnet.layer1)
        self.layer2 = resnet.layer2  # stride  8, 38x38, channels  512
        self.layer3 = resnet.layer3  # stride 16, 19x19, channels 1024
        self.layer4 = resnet.layer4  # stride 32, 10x10, channels 2048

        # さらに解像度を落とす追加の畳み込み層（SSDの深い特徴マップに対応）
        self.extra1 = nn.Sequential(
            nn.Conv2d(2048, 512, kernel_size=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, stride=2, padding=1), nn.ReLU(inplace=True))  # 10x10 -> 5x5
        self.extra2 = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, stride=2, padding=1), nn.ReLU(inplace=True))  # 5x5 -> 3x3
        self.extra3 = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3), nn.ReLU(inplace=True))  # 3x3 -> 1x1

        self.out_channels = [512, 1024, 2048, 512, 256, 256]

    def forward(self, x):
        x = self.stem(x)
        c3 = self.layer2(x)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        c6 = self.extra1(c5)
        c7 = self.extra2(c6)
        c8 = self.extra3(c7)
        return [c3, c4, c5, c6, c7, c8]

### デフォルトボックス（Prior Box）の生成
各特徴マップの各セル（格子点）に対して，複数のアスペクト比・スケールを持つデフォルトボックスを配置します．本ノートブックでは，実装を単純化するため，6つの特徴マップすべてで共通のアスペクト比集合 $\{1, 2, 1/2, 3, 1/3\}$ に，アスペクト比1でスケールが少し大きいボックスを1つ加えた，合計6種類のデフォルトボックスを各セルに配置します（原論文では，特徴マップごとにボックスの種類数が4種類または6種類と異なります）．

デフォルトボックスの大きさ（スケール）は，浅い特徴マップほど小さく，深い特徴マップほど大きくなるように，原論文の式に従って線形に決定します．座標は，画像サイズに対する比率（0〜1の正規化座標）の中心座標・幅・高さ（cx, cy, w, h）で表します．

In [ ]:
FEATURE_MAP_SIZES = [38, 19, 10, 5, 3, 1]
ASPECT_RATIOS = [1, 2, 1 / 2, 3, 1 / 3]
MIN_SCALE, MAX_SCALE = 0.1, 0.9
SCALES = [MIN_SCALE + (MAX_SCALE - MIN_SCALE) * i / (len(FEATURE_MAP_SIZES) - 1) for i in range(len(FEATURE_MAP_SIZES))]
SCALES.append(1.0)  # 最後の特徴マップの「1つ先のスケール」として1.0を追加（アスペクト比1の追加ボックス用）


def generate_default_boxes():
    boxes = []
    for k, f in enumerate(FEATURE_MAP_SIZES):
        for i in range(f):
            for j in range(f):
                cx = (j + 0.5) / f
                cy = (i + 0.5) / f

                for ar in ASPECT_RATIOS:
                    w = SCALES[k] * (ar ** 0.5)
                    h = SCALES[k] / (ar ** 0.5)
                    boxes.append([cx, cy, w, h])

                # アスペクト比1で，隣接するスケールとの幾何平均を大きさに持つ追加ボックス
                extra_scale = (SCALES[k] * SCALES[k + 1]) ** 0.5
                boxes.append([cx, cy, extra_scale, extra_scale])

    return torch.tensor(boxes, dtype=torch.float32).clamp(min=0.0, max=1.0)


default_boxes = generate_default_boxes().to(device)  # (num_boxes, 4), (cx, cy, w, h) の正規化座標
default_boxes_xyxy = (box_convert(default_boxes, 'cxcywh', 'xyxy').clamp(0, 1) * IMG_SIZE)  # ピクセル座標(xyxy)
num_boxes_per_cell = len(ASPECT_RATIOS) + 1
print('デフォルトボックスの総数:', default_boxes.shape[0])

### ボックスのエンコード・デコード
学習時には，正解ボックス（Ground Truth）をそのまま回帰対象にするのではなく，マッチしたデフォルトボックスからの相対的なずれ（オフセット）に変換（エンコード）してから学習します．推論時には，ネットワークが出力したオフセットを，対応するデフォルトボックスに適用して画像上の座標に戻します（デコード）．`variances`は，オフセットのスケールを調整するための定数です．

In [ ]:
def encode_boxes(matched_boxes_xyxy, variances=(0.1, 0.2)):
    """マッチしたGTボックス（xyxy, ピクセル座標）を，デフォルトボックスからのオフセットにエンコードする"""
    matched_cxcywh = box_convert(matched_boxes_xyxy / IMG_SIZE, 'xyxy', 'cxcywh')
    g_cxcy = (matched_cxcywh[:, :2] - default_boxes[:, :2]) / (default_boxes[:, 2:] * variances[0])
    g_wh = torch.log(matched_cxcywh[:, 2:] / default_boxes[:, 2:] + 1e-8) / variances[1]
    return torch.cat([g_cxcy, g_wh], dim=1)


def decode_boxes(loc_preds, variances=(0.1, 0.2)):
    """ネットワークが出力したオフセットを，画像上のボックス座標（xyxy, ピクセル座標）にデコードする"""
    cxcy = loc_preds[..., :2] * variances[0] * default_boxes[:, 2:] + default_boxes[:, :2]
    wh = torch.exp(loc_preds[..., 2:] * variances[1]) * default_boxes[:, 2:]
    boxes_cxcywh = torch.cat([cxcy, wh], dim=-1)
    boxes_xyxy = box_convert(boxes_cxcywh, 'cxcywh', 'xyxy') * IMG_SIZE
    return boxes_xyxy.clamp(min=0, max=IMG_SIZE)  # 画像範囲外に出たボックスをクリップする

### デフォルトボックスとGTボックスのマッチング
学習時には，各デフォルトボックスに対して，どのGTボックス（あるいは背景）を教師信号とするかを決める必要があります．本ノートブックでは，原論文と同様に次の2段階でマッチングを行います．

1. 各GTボックスに対して，IoU（Intersection over Union）が最大となるデフォルトボックスを，必ず正例としてそのGTに割り当てる．
2. さらに，IoUが閾値（0.5）以上となる残りのデフォルトボックスも，IoU最大のGTに割り当てて正例とする．
3. どちらの条件も満たさないデフォルトボックスは，クラス0（背景）とする．

IoUの計算には`torchvision.ops.box_iou`を用います．

In [ ]:
def match_default_boxes(gt_boxes, gt_labels, iou_threshold=0.5):
    """1枚の画像内のGTボックスをデフォルトボックスに割り当て，学習用のラベル・オフセットターゲットを作成する"""
    num_default = default_boxes.shape[0]
    if gt_boxes.shape[0] == 0:
        return torch.zeros(num_default, dtype=torch.long, device=device), torch.zeros(num_default, 4, device=device)

    ious = box_iou(gt_boxes, default_boxes_xyxy)  # (num_gt, num_default)

    # 各デフォルトボックスに対して，最もIoUが高いGTを割り当てる
    best_gt_iou, best_gt_idx = ious.max(dim=0)  # (num_default,)
    # 各GTボックスに対して，最もIoUが高いデフォルトボックスは，閾値によらず必ず正例にする
    _, best_default_idx = ious.max(dim=1)  # (num_gt,)
    best_gt_iou[best_default_idx] = 1.0
    best_gt_idx[best_default_idx] = torch.arange(gt_boxes.shape[0], device=gt_boxes.device)

    labels = gt_labels[best_gt_idx].clone()
    labels[best_gt_iou < iou_threshold] = 0  # 背景

    matched_boxes = gt_boxes[best_gt_idx]
    loc_targets = encode_boxes(matched_boxes)

    return labels, loc_targets

## 損失関数（MultiBox Loss）
SSDの損失は，物体の位置回帰に対する**Localization Loss**（Smooth L1損失，正例のデフォルトボックスのみで計算）と，クラス分類に対する**Confidence Loss**（クロスエントロピー損失）の和で構成されます．

デフォルトボックスの大部分は背景（負例）であるため，そのまま全ての負例を使って学習すると，背景クラスの学習ばかりが支配的になってしまいます．そこで，**Hard Negative Mining**という手法を用い，各画像について損失の大きい負例を，正例の数の`neg_pos_ratio`倍（本ノートブックでは3倍）だけ選んで学習に使用します．

In [ ]:
class MultiBoxLoss(nn.Module):
    def __init__(self, neg_pos_ratio=3):
        super().__init__()
        self.neg_pos_ratio = neg_pos_ratio
        self.smooth_l1 = nn.SmoothL1Loss(reduction='sum')

    def forward(self, loc_preds, conf_preds, loc_targets, labels):
        pos_mask = labels > 0
        num_pos_total = pos_mask.sum().clamp(min=1)

        # Localization Loss: 正例のデフォルトボックスのみで計算
        loc_loss = self.smooth_l1(loc_preds[pos_mask], loc_targets[pos_mask])

        # Confidence Loss: Hard Negative Miningにより，正例:負例が1:neg_pos_ratioになるよう負例を選択
        conf_loss_all = F.cross_entropy(conf_preds.view(-1, conf_preds.size(-1)), labels.view(-1), reduction='none')
        conf_loss_all = conf_loss_all.view(labels.shape)

        conf_loss_pos = conf_loss_all[pos_mask].sum()

        conf_loss_neg = conf_loss_all.clone()
        conf_loss_neg[pos_mask] = -1.0  # 正例は負例マイニングの対象から除外（順位付けで最下位になるようにする）
        num_neg = (pos_mask.sum(dim=1, keepdim=True) * self.neg_pos_ratio).clamp(max=labels.size(1) - 1)
        _, neg_rank = conf_loss_neg.sort(dim=1, descending=True)
        _, neg_rank_idx = neg_rank.sort(dim=1)
        neg_mask = neg_rank_idx < num_neg

        conf_loss_neg_selected = conf_loss_all[neg_mask].sum()

        return (loc_loss + conf_loss_pos + conf_loss_neg_selected) / num_pos_total

## ネットワークの作成
特徴マップごとに，デフォルトボックス1つあたり「4次元のオフセット」と「`n_class`クラス分のスコア」を出力する畳み込み層（`SSDHead`）を定義し，バックボーンと組み合わせて`SSD`モデルを構築します．出力は，全ての特徴マップ・全てのデフォルトボックス分を1つのテンソルに連結して返します．

`.to(device)`によりモデルのパラメータを`device`上に配置し，最適化手法にはモーメンタムSGD（学習率0.001，モーメンタム0.9，weight decay 5e-4）を用います．事前学習済みのバックボーンを使用するため，スクラッチ学習のCIFAR100分類モデルよりも小さめの学習率を設定しています．

In [ ]:
class SSDHead(nn.Module):
    def __init__(self, in_channels_list, num_boxes_per_cell, n_class):
        super().__init__()
        self.loc_layers = nn.ModuleList([
            nn.Conv2d(c, num_boxes_per_cell * 4, kernel_size=3, padding=1) for c in in_channels_list
        ])
        self.conf_layers = nn.ModuleList([
            nn.Conv2d(c, num_boxes_per_cell * n_class, kernel_size=3, padding=1) for c in in_channels_list
        ])
        self.n_class = n_class

    def forward(self, features):
        locs, confs = [], []
        for feat, loc_layer, conf_layer in zip(features, self.loc_layers, self.conf_layers):
            loc = loc_layer(feat).permute(0, 2, 3, 1).contiguous().view(feat.size(0), -1, 4)
            conf = conf_layer(feat).permute(0, 2, 3, 1).contiguous().view(feat.size(0), -1, self.n_class)
            locs.append(loc)
            confs.append(conf)
        return torch.cat(locs, dim=1), torch.cat(confs, dim=1)


class SSD(nn.Module):
    def __init__(self, n_class, num_boxes_per_cell):
        super().__init__()
        self.backbone = SSDBackbone()
        self.head = SSDHead(self.backbone.out_channels, num_boxes_per_cell, n_class)

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)


model = SSD(n_class=n_class, num_boxes_per_cell=num_boxes_per_cell)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=5e-4)
criterion = MultiBoxLoss(neg_pos_ratio=3)

n_params = sum(p.numel() for p in model.parameters())
print('総パラメータ数:', n_params)

## 学習
読み込んだVOC2007データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを8，学習エポック数を20とします．画像1枚あたりの物体数が異なるため，`collate_fn`でバッチを作成し，各画像のGTボックスをバッチ内でループしながらデフォルトボックスとマッチングさせます。

In [ ]:
batch_size = 8
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=collate_fn)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0

    for images, targets in train_loader:
        images = images.to(device)

        batch_labels, batch_loc_targets = [], []
        for t in targets:
            labels, loc_targets = match_default_boxes(t['boxes'].to(device), t['labels'].to(device))
            batch_labels.append(labels)
            batch_loc_targets.append(loc_targets)
        batch_labels = torch.stack(batch_labels)
        batch_loc_targets = torch.stack(batch_loc_targets)

        loc_preds, conf_preds = model(images)

        loss = criterion(loc_preds, conf_preds, batch_loc_targets, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()

    print("epoch: {}, mean loss: {}, elapsed time: {}".format(epoch, sum_loss / len(train_loader), time() - train_start))

## 推論（デコード・NMS）
学習したモデルで推論を行うための関数を定義します．出力されたクラススコアにSoftmaxを適用し，クラスごとに信頼度が閾値を超えるデフォルトボックスをデコードしたのち，`torchvision.ops.nms`でクラスごとに重複した検出結果を除去（Non-Maximum Suppression）します。

In [ ]:
def predict(model, image_tensor, conf_threshold=0.5, nms_threshold=0.45):
    model.eval()
    with torch.no_grad():
        loc_preds, conf_preds = model(image_tensor.unsqueeze(0).to(device))
    scores_all = F.softmax(conf_preds[0], dim=-1)
    boxes_all = decode_boxes(loc_preds[0])

    result_boxes, result_scores, result_labels = [], [], []
    for class_idx in range(1, n_class):  # 背景（クラス0）は除外
        class_scores = scores_all[:, class_idx]
        mask = class_scores > conf_threshold
        if mask.sum() == 0:
            continue
        class_boxes = boxes_all[mask]
        class_scores = class_scores[mask]

        keep = nms(class_boxes, class_scores, nms_threshold)
        result_boxes.append(class_boxes[keep])
        result_scores.append(class_scores[keep])
        result_labels.append(torch.full((len(keep),), class_idx, dtype=torch.long))

    if len(result_boxes) == 0:
        return torch.zeros(0, 4), torch.zeros(0), torch.zeros(0, dtype=torch.long)

    return torch.cat(result_boxes).cpu(), torch.cat(result_scores).cpu(), torch.cat(result_labels).cpu()

## テスト
評価指標には，物体検出タスクで広く使われる**mAP（mean Average Precision）**を用います．mAPの算出は，IoUの閾値判定やクラスごとのPrecision-Recall曲線の計算など実装が煩雑なため，`torchmetrics`の`MeanAveragePrecision`を利用します．VOCで伝統的に使われるIoU閾値0.5でのmAP（`map_50`）を確認します。

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1, shuffle=False, collate_fn=collate_fn)

metric = MeanAveragePrecision(iou_type='bbox')
model.eval()

for images, targets in test_loader:
    boxes, scores, labels = predict(model, images[0])

    preds = [{'boxes': boxes, 'scores': scores, 'labels': labels}]
    gts = [{'boxes': targets[0]['boxes'], 'labels': targets[0]['labels']}]
    metric.update(preds, gts)

result = metric.compute()
print('mAP@0.5:', result['map_50'].item())
print('mAP@[0.5:0.95]:', result['map'].item())

## 検出結果の可視化
テストデータ1枚に対する検出結果を，`torchvision.utils.draw_bounding_boxes`を用いて画像上に描画します。

In [ ]:
image, target = test_data[0]
boxes, scores, labels = predict(model, image)

# 正規化を戻して0-255のuint8画像に変換
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img_show = (image * std + mean).clamp(0, 1)
img_show = (img_show * 255).to(torch.uint8)

label_names = [f'{VOC_CLASSES[l - 1]}: {s:.2f}' for l, s in zip(labels, scores)]
img_with_boxes = draw_bounding_boxes(img_show, boxes, labels=label_names, colors='red', width=2)

plt.figure(figsize=(6, 6))
plt.imshow(img_with_boxes.permute(1, 2, 0))
plt.axis('off')
plt.show()

## オリジナルSSDとの違い
本ノートブックで実装したSSDは，原論文の構成を一部簡略化しています．主な違いは次の通りです．

| | 原論文（SSD300） | 本ノートブック |
|---|---|---|
| バックボーン | VGG16 | ResNet50（ImageNet事前学習済み） |
| 入力画像サイズ | 300×300 | 300×300（変更なし） |
| 特徴マップのサイズ | 38, 19, 10, 5, 3, 1 | 38, 19, 10, 5, 3, 1（変更なし） |
| 特徴マップごとのデフォルトボックス数 | 4, 6, 6, 6, 4, 4（特徴マップごとに異なる） | 全ての特徴マップで6種類に統一 |
| NMS・IoU計算 | 独自実装 | `torchvision.ops`の`nms`・`box_iou`を使用 |

特徴マップの構成（各段階の解像度）自体は原論文と同じですが，バックボーンをVGG16からResNet50に変更し，デフォルトボックスの構成を単純化している点が主な変更点です。

## 課題

1. `conf_threshold`や`nms_threshold`を変更し，検出結果（検出数・誤検出・重複検出）がどのように変化するか確認しましょう．

2. `MultiBoxLoss`の`neg_pos_ratio`を変更し，Hard Negative Miningの効果や学習の安定性への影響を確認しましょう．

3. `ASPECT_RATIOS`や特徴マップごとのデフォルトボックス数を，原論文の設定（浅い/深い特徴マップは4種類，中間の特徴マップは6種類）に近づけて実装し直し，パラメータ数や検出精度への影響を確認しましょう．